# Data Science & Database Assignment: Session 3
## Topic: INSERT, UPDATE & DELETE

**Student Name:** Atharv Chaudhari  
**Course:** Data Science & Database Fundamentals  
**Database Name:** `playlist_app`  
**Core Concepts:** Data Manipulation Language (DML) — `INSERT INTO`, `UPDATE ... SET ... WHERE`, `DELETE FROM ... WHERE`, String Concatenation, Compound Logical Operators (`AND`)  
**Status:** Completed & Pre-Executed  

---

### DML Lifecycle & The Critical Importance of `WHERE`

In SQL, **DML (Data Manipulation Language)** controls the data stored inside database tables:
1. **`INSERT INTO`**: Adds new rows to a table.
2. **`UPDATE`**: Modifies existing values in one or more columns.
3. **`DELETE`**: Permanently removes rows from a table.

> ⚠️ **The Golden Rule of DML Operations:**  
> Always verify your `WHERE` filter condition before running `UPDATE` or `DELETE` in production. Omitting the `WHERE` clause applies the change or deletion to **every single row** in the table!


## Task 1: Create `Playlist` Table & Insert Single Favorite Song
> **Problem Statement:** Create a table called `Playlist` with columns: `id` (INT, primary key), `song_name` (VARCHAR), `artist` (VARCHAR), and `duration` (INT, seconds). Insert a single row for your current favorite song.

### SQL Statements:
```sql
CREATE TABLE Playlist (
    id INT PRIMARY KEY,
    song_name VARCHAR(255) NOT NULL,
    artist VARCHAR(255) NOT NULL,
    duration INT NOT NULL
);

INSERT INTO Playlist (id, song_name, artist, duration)
VALUES (1, 'Tum Hi Ho', 'Arjit Singh', 262);
```
*(Note: We intentionally include the typo `'Arjit Singh'` to be corrected in Task 3).*


In [1]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect("playlist_app.db")
cursor = conn.cursor()

# Drop existing table for clean rerun
cursor.execute("DROP TABLE IF EXISTS Playlist;")

# Create Playlist table
cursor.execute("""
CREATE TABLE Playlist (
    id INTEGER PRIMARY KEY,
    song_name VARCHAR(255) NOT NULL,
    artist VARCHAR(255) NOT NULL,
    duration INTEGER NOT NULL
);
""")

# Insert single favorite song
cursor.execute("""
INSERT INTO Playlist (id, song_name, artist, duration)
VALUES (1, 'Tum Hi Ho', 'Arjit Singh', 262);
""")
conn.commit()

df_t1 = pd.read_sql_query("SELECT * FROM Playlist;", conn)
print("Task 1 Output: Playlist with Single Favorite Song")
display(df_t1)


Task 1 Output: Playlist with Single Favorite Song


,id,song_name,artist,duration
0,1,Tum Hi Ho,Arjit Singh,262


## Task 2: Insert 3 Recently Listened Spotify Songs
> **Problem Statement:** Insert 3 new rows into the `Playlist` table for songs you recently listened to on Spotify, including their `song_name`, `artist`, and `duration`.

### SQL Statements:
```sql
INSERT INTO Playlist (id, song_name, artist, duration) VALUES
(2, 'Brown Munde', 'AP Dhillon', 260),
(3, 'Summer High', 'AP Dhillon', 175),
(4, 'Short Interlude', 'The Weeknd', 90);
```


In [2]:
new_songs = [
    (2, 'Brown Munde', 'AP Dhillon', 260),
    (3, 'Summer High', 'AP Dhillon', 175),
    (4, 'Short Interlude', 'The Weeknd', 90)
]

cursor.executemany("""
INSERT INTO Playlist (id, song_name, artist, duration)
VALUES (?, ?, ?, ?);
""", new_songs)
conn.commit()

df_t2 = pd.read_sql_query("SELECT * FROM Playlist;", conn)
print(f"Task 2 Output: Playlist with 3 Newly Added Songs (Total: {len(df_t2)})")
display(df_t2)


Task 2 Output: Playlist with 3 Newly Added Songs (Total: 4)


,id,song_name,artist,duration
0,1,Tum Hi Ho,Arjit Singh,262
1,2,Brown Munde,AP Dhillon,260
2,3,Summer High,AP Dhillon,175
3,4,Short Interlude,The Weeknd,90


## Task 3: Fix Artist Typo Using `UPDATE ... WHERE`
> **Problem Statement:** Update the artist name for one of your `Playlist` entries to fix a typo (for example, change `'Arjit Singh'` to `'Arijit Singh'`) using the `UPDATE` statement with a `WHERE` clause.

### SQL Query & Breakdown:
```sql
UPDATE Playlist
SET artist = 'Arijit Singh'
WHERE artist = 'Arjit Singh';
```
* **`UPDATE Playlist`**: Specifies the target table.
* **`SET artist = 'Arijit Singh'`**: Assigns the corrected string to the column.
* **`WHERE artist = 'Arjit Singh'`**: Precise filter ensuring only the typo record is modified without overwriting other artists.


In [3]:
# Execute UPDATE query
update_typo_sql = """
UPDATE Playlist
SET artist = 'Arijit Singh'
WHERE artist = 'Arjit Singh';
"""
cursor.execute(update_typo_sql)
conn.commit()

print(f"Rows updated: {cursor.rowcount}")

df_t3 = pd.read_sql_query("SELECT * FROM Playlist;", conn)
print("Task 3 Output: Playlist After Typo Fix ('Arjit Singh' -> 'Arijit Singh')")
display(df_t3)


Rows updated: 1
Task 3 Output: Playlist After Typo Fix ('Arjit Singh' -> 'Arijit Singh')


,id,song_name,artist,duration
0,1,Tum Hi Ho,Arijit Singh,262
1,2,Brown Munde,AP Dhillon,260
2,3,Summer High,AP Dhillon,175
3,4,Short Interlude,The Weeknd,90


## Task 4: Delete Songs Shorter Than 120 Seconds (`DELETE ... WHERE`)
> **Problem Statement:** Delete a song from the `Playlist` table where the duration is less than 120 seconds using the `DELETE` statement and a `WHERE` clause.  
> *Hint: Make sure your `WHERE` clause is specific so you don't accidentally delete all rows.*

### SQL Query & Safety Precaution:
```sql
DELETE FROM Playlist
WHERE duration < 120;
```
* **Target Record:** `'Short Interlude'` (90 seconds).
* **Safe Practice:** In production environments, it is recommended to test your filter first with a `SELECT` statement before deleting:
  ```sql
  -- Step 1 (Preview): SELECT * FROM Playlist WHERE duration < 120;
  -- Step 2 (Execute): DELETE FROM Playlist WHERE duration < 120;
  ```


In [4]:
# Preview targeted rows
preview_delete = pd.read_sql_query("SELECT * FROM Playlist WHERE duration < 120;", conn)
print("Rows matching deletion criteria (duration < 120s):")
display(preview_delete)

# Execute DELETE statement
cursor.execute("DELETE FROM Playlist WHERE duration < 120;")
conn.commit()
print(f"\nSuccessfully deleted {cursor.rowcount} row(s).")

df_t4 = pd.read_sql_query("SELECT * FROM Playlist;", conn)
print("Task 4 Output: Playlist After Deleting Short Tracks")
display(df_t4)


Rows matching deletion criteria (duration < 120s):


,id,song_name,artist,duration
0,4,Short Interlude,The Weeknd,90



Successfully deleted 1 row(s).
Task 4 Output: Playlist After Deleting Short Tracks


,id,song_name,artist,duration
0,1,Tum Hi Ho,Arijit Singh,262
1,2,Brown Munde,AP Dhillon,260
2,3,Summer High,AP Dhillon,175


## Task 5: Compound Conditional UPDATE (`AP Dhillon` & Duration > 180s)
> **Problem Statement:** Write an SQL statement that would update the `song_name` for all songs by `'AP Dhillon'` in your `Playlist` to add `'(Remix)'` at the end of the name, but only if the duration is more than 180 seconds.  
> *Constraint: Combine UPDATE with WHERE to target only the correct rows.*

### SQL Statement:
```sql
-- Standard SQL / SQLite / PostgreSQL:
UPDATE Playlist
SET song_name = song_name || ' (Remix)'
WHERE artist = 'AP Dhillon' AND duration > 180;

-- MySQL Equivalent:
-- UPDATE Playlist
-- SET song_name = CONCAT(song_name, ' (Remix)')
-- WHERE artist = 'AP Dhillon' AND duration > 180;
```

### Analysis of Targeted Rows:
* `'Brown Munde'` (artist = `'AP Dhillon'`, duration = `260s` > 180) $ightarrow$ **UPDATED to `'Brown Munde (Remix)'`**
* `'Summer High'` (artist = `'AP Dhillon'`, duration = `175s` $\le$ 180) $ightarrow$ **UNCHANGED**
* `'Tum Hi Ho'` (artist = `'Arijit Singh'`) $ightarrow$ **UNCHANGED**


In [5]:
# Execute compound UPDATE
update_remix_sql = """
UPDATE Playlist
SET song_name = song_name || ' (Remix)'
WHERE artist = 'AP Dhillon' AND duration > 180;
"""
cursor.execute(update_remix_sql)
conn.commit()

print(f"Rows matching compound condition updated: {cursor.rowcount}")

df_t5 = pd.read_sql_query("SELECT * FROM Playlist;", conn)
print("Task 5 Output: Final Playlist State with Targeted Remix Update")
display(df_t5)

# Clean up connection
conn.close()


Rows matching compound condition updated: 1
Task 5 Output: Final Playlist State with Targeted Remix Update


,id,song_name,artist,duration
0,1,Tum Hi Ho,Arijit Singh,262
1,2,Brown Munde (Remix),AP Dhillon,260
2,3,Summer High,AP Dhillon,175


---

## 📌 Summary of Core DML Statements Mastered

| Operation | SQL Command Pattern | Safety Precaution |
| :--- | :--- | :--- |
| **Insert Single Row** | `INSERT INTO table (cols) VALUES (vals);` | Ensure data types match column definitions |
| **Insert Multi-Row** | `INSERT INTO table VALUES (...), (...);` | Faster ingestion via batched insert |
| **Update Single Field** | `UPDATE table SET col = new_val WHERE condition;` | Always provide specific `WHERE` filter |
| **String Concatenation**| `SET col = col || ' (suffix)'` (or `CONCAT()`) | Preserves original content while appending |
| **Delete Specific Rows**| `DELETE FROM table WHERE condition;` | Run `SELECT` first to verify targeted rows |
